This code detects the location of starting point of an office;

Input: List of offices + Corresponding offices 

=> Output: List of offices with coordinate information

In [1]:
import pandas as pd
import os
import cv2
import numpy as np
import json

In [2]:
def Detect_Office(Json,Office):

    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if (Office[0] == d['inferText'][0]) or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

def Detect_Office2(Json,Office):
    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if ( d['inferText'][0] == '〇') or( d['inferText'][0] == '○') or ( d['inferText'][0] == '0')or ( d['inferText'][0] == 'O') or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

### CLOVA FUNCTION ###
import requests
import uuid
import time
import json
import cv2
import base64

api_url = 'https://deelieyxuc.apigw.ntruss.com/custom/v1/1972/ebd01bcbf693d069817622e9839e20914143c7d0d8953eddee40e8b0af96c95b/general'
secret_key = 'S1NmVXpYZlJ0cGJ0ZEFRZXVlbkRkaHFReE9FcHNTQ0U='

def Clova(file_data):
    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

def Clova2(Year,Page):
    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
    stream = open(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg", "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    HH,WW=img.shape[:2]
    cropped=img[0:200,0:WW]
    temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
    cv2.imwrite(temp_path+"Temp.jpg",cropped)
    
    with open(temp_path+"Temp.jpg",'rb') as f:
        file_data = f.read()

    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

In [3]:
def hconcat_resize_min(im_list, interpolation=cv2.INTER_CUBIC):
    h_min = min(im.shape[0] for im in im_list)
    im_list_resize = [cv2.resize(im, (int(im.shape[1] * h_min / im.shape[0]), h_min), interpolation=interpolation)
                      for im in im_list]
    return cv2.hconcat(im_list_resize)

def Check(df):
    for n in range(0,len(df)):
        #Extract key info of office
        Row  = df.iloc[n]

        ###Collect Location information###
        Dept=Row["Dept"]
        Office=Row["Office"]
        print('['+str(n)+',"'+Dept+'","'+Office+'"]')
        try:        
            StrPage=dta[Year][Dept][Office]['StrPage']
            StrPart=dta[Year][Dept][Office]['StrPart']
            EndPage=dta[Year][Dept][Office]['EndPage']

            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+StrPart+".jpg",'rb')
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
            HH,WW=img.shape[:2]
            oldimg=img.copy()[0:200,0:WW]

            for Page in range(StrPage+1,EndPage+1):
                path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
                stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+Part+".jpg",'rb') 
                bytes = bytearray(stream.read())
                numpyarray = np.asarray(bytes, dtype=np.uint8)
                newimg = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)[0:200,0:WW]
                oldimg=np.concatenate((newimg,oldimg), axis=1)
            oldimg=cv2.copyMakeBorder(oldimg,20,20,20,20,cv2.BORDER_CONSTANT)

            StrLoc=int(dta[Year][Dept][Office]['Office_X1'])
            EndLoc=int(dta[Year][Dept][Office]['Office_X2'])
            StrPage=int(dta[Year][Dept][Office]['Starting_Page'])
            EndPage=int(dta[Year][Dept][Office]['Ending_Page'])

            #Start#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+"Page"+"{:03d}".format(StrPage)+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(StrLoc,0),(StrLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            #End#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(EndPage)+"\\"+"Page"+"{:03d}".format(EndPage)+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(EndLoc,0),(EndLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            cv2.imshow(Dept+Office,oldimg)
            cv2.waitKey(0)
        except:
            continue
    cv2.destroyAllWindows()
    cv2.waitKey(0)

In [4]:
def Show(n,StrLoc):
    Row=df.iloc[n]
    Page=Row['Page']
    
    path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
    stream = open(path+"Page"+"{:03d}".format(EndPage)+"\\"+"Page"+"{:03d}".format(EndPage)+".jpg",'rb')
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    
    HH,WW=img.shape[:2]
    cv2.line(img,(StrLoc,0),(StrLoc,HH),(225,0,0),2)
    cv2.imshow('Projection',img)
    cv2.waitKey(0)

#Step 1

Fill in years to refer.
Remember to keep it as a string. NOT as float.

In [6]:
Year='1940'
Showa='15'
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
os.chdir(path)
df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

file_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+Year+'\\DataFrame.json'
with open(file_path, encoding="utf-8") as f:
    dta = json.loads(f.read())

#Step 2

The following code will extract the location of offices.

In [7]:
n=0
#Extract key info of office
Row  = df.iloc[n]

Page=int(Row["Page"])
Office=Row["Office"]
Dept=Row['Dept']

###Insert Starting page information to motherframe###
dta[Year][Dept][Office]={}
dta[Year][Dept][Office]["Starting_Page"]=Page
print(dta[Year][Dept][Office])

###Collect Location information###
#Part1
with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
    file_data = f.read()
#Convert to json via CLOVA
Json=Clova(file_data)

#Find X coordinate of 'Office'.
XCoord_Unit=Detect_Office(Json,Office)
if XCoord_Unit==None:
    #Add to motherframe
    dta[str(Year)][Dept][Office]["StrLocation"]='NA'
else:
    dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
    dta[str(Year)][Dept][Office]["StrPart"]='Top'
    HH=img.shape[:2][0]
    print(Office+ 'success: at row'+str(n))

#Part2
with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
    file_data = f.read()
#Convert to json via CLOVA
Json=Clova(file_data)

#Find X coordinate of 'Office'.
XCoord_Unit=Detect_Office(Json,Office)
if XCoord_Unit==None:
    #Add to motherframe
    dta[str(Year)][Dept][Office]["StrLocation"]='NA'
    print(Office+ 'failed')
    FailedList.append(list((n,Dept,Office)))
else:
    dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
    dta[str(Year)][Dept][Office]["StrPart"]='Btm'
    print(Office+ 'success: at row'+str(n))

{'Starting_Page': 3}
秘書課success: at row0


In [8]:
#Test code| Version 2#
#Show Working office#
FailedList=[]
for n in range(1,len(df)):
    #Extract key info of office
    Row  = df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']
    print('['+str(n)+',"'+Dept+'","'+Office+'"]')
    
    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Insert Starting page information to motherframe###
    dta[Year][Dept][Office]={}
    dta[Year][Dept][Office]["StrPage"]=Page
    print(dta[Year][Dept][Office])

    ###Collect Location information###
    #Part1
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
        file_data = f.read()
    #Convert to json via CLOVA
    try:
        Json=Clova(file_data)
    except:
        print(Office+ 'failed')
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    print('Top Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))
    
    if XCoord_Unit!=None:
        continue
        
    #Part2
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
        file_data = f.read()

    #Convert to json via CLOVA
    try:
        Json=Clova(file_data)
    except:
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    print('Bottom Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))

[1,"総務局","文書課"]
{'StrPage': 3}
Top LocNone
Bottom Loc176
文書課success: at row1
[2,"総務局","人事課"]
{'StrPage': 5}
Top Loc459
人事課success: at row2
[3,"総務局","吏務課"]
{'StrPage': 5}
Top Loc106
吏務課success: at row3
[4,"総務局","議案課"]
{'StrPage': 5}
Top LocNone
Bottom Loc93
議案課success: at row4
[5,"総務局","企画課"]
{'StrPage': 6}
Top Loc168
企画課success: at row5
[6,"総務局","都市計画課"]
{'StrPage': 6}
Top LocNone
Bottom Loc36
都市計画課success: at row6
[7,"総務局","統計課"]
{'StrPage': 8}
Top Loc288
統計課success: at row7
[8,"総務局","情報課（S14.12.12）"]
{'StrPage': 10}
Top Loc394
情報課（S14.12.12）success: at row8
[9,"総務局","市務監察課"]
{'StrPage': 10}
Top LocNone
Bottom LocNone
市務監察課failed
[10,"総務局","区務監察課"]
{'StrPage': 11}
Top LocNone
Bottom LocNone
区務監察課failed
[11,"財務局","会計課"]
{'StrPage': 11}
Top LocNone
Bottom Loc245
会計課success: at row11
[12,"財務局","主計課"]
{'StrPage': 12}
Top LocNone
Bottom Loc301
主計課success: at row12
[13,"財務局","主税課"]
{'StrPage': 13}
Top Loc236
主税課success: at row13
[14,"財務局","公債課"]
{'StrPage': 16}
Top Loc432
公債課success: at row

#Step 3

Check for errors.
Manually type in the index, department name, and office name of erroneous department-office pair to a seperate list.

In [ ]:
Check(df)

Type in the department-offices with errors.

Type1=[]

#Step 4

See how much errors were observed in the first trial.

In [11]:
FailRate=len(FailedList)/len(df)
if len(FailedList)/len(df)<0.2:
    print('Fantastic!! Success Rate is '+str(1-(len(FailedList)/len(df))))
else:
    print('残念...Success Rate is '+str(1-(len(FailedList)/len(df))))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')
DF.loc[int(Year)-1934, 'Office'] = 1-FailRate
display(DF)
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv',index=False)

Fantastic!! Success Rate is 0.8518518518518519


,Year,WritePage,Page,Office,Unit,Position,Data
0,1934,1,0.985714286,0.9,.,0.08,.
1,1935,1,1,0.984375,.,-0.09375,.
2,1936,0.235294118,.,0.235294118,.,.,.
3,1937,1,1,0.865671642,.,.,.
4,1938,1,1,1,0.988095238,0.98816568,0.619957537
5,1939,1,1,1,.,.,.
6,1940,1,.,0.851852,.,.,.
7,1941,.,.,.,.,.,.
8,1942,.,.,.,.,.,.
9,1943,.,.,.,.,.,.


In [27]:
#Fixing Failed Offices
#Step1: Check for simple page assignment error
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
for n in range(0,len(FailedList)):
    Dept=FailedList[n][1]
    Office=FailedList[n][2]
    print(Dept,Office)
    Page=df['Page'][(df['Office']==Office) & (df['Dept']==Dept)].tolist()[0]
    print(Page)
    img1=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\Top.jpg")
    img2=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\Btm.jpg")
    cv2.imshow('Top',img1)
    cv2.waitKey(0)
    cv2.imshow('Btm',img2)
    cv2.waitKey(0)
    

総務局 市務監察課
10
総務局 区務監察課
11
財務局 学校営業課
22
市民局 第一課
28
市民局 第二課
29
市民局 計画課
28
教育局 学校職員課
30
教育局 視学課
30
教育局 社会教育課
32
厚生局 防疫課
49
厚生局 医務課
60
厚生局 分院及学校
60
土木局 治水工事課
85
土木局 区土木課役務者
86
水道局 計書課
105
電気局 臨時電源調査課
137


Fix the department-office pair with errors

First re-run the list of failed (pairs which was rejected), with a stricter OCR.

In [ ]:
FailedList2=[]
for Office in FailedList:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Collect Location information###
    #Part1
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
        file_data = f.read()
    #Convert to json via CLOVA
    try:
        Json=Clova(file_data)
    except:
        print(Office+ 'failed')
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    print('Top Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))
    
    if XCoord_Unit!=None:
        continue
        
    #Part2
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
        file_data = f.read()

    #Convert to json via CLOVA
    try:
        Json=Clova(file_data)
    except:
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    print('Bottom Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))

In [ ]:
FailRate=len(FailedList2)/len(df)
if FailRate<0.2:
    print('Fantastic!! Success Rate is '+str(1-FailRate))
else:
    print('残念...Success Rate is '+str(1-FailRate))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')
DF.loc[int(Year)-1934, 'Office'] = 1-FailRate
display(DF)
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv',index=False)

Open FailedList_2 and see if there are any other errors left.

If there are, fix it manually by guessing the starting x axis using 'Show(n,StrLoc)'

When you find the exact starting location, update the dataframe.

dta[Year][Dept][Office]={'Starting_Page': ,
                          'Office_X1': ,
                          'Ending_Page': ,
                          'Office_X2': ,
                          'Page_Range': []}

#Step 5

Do the same thing with the department-office pair which was wrongly accepted in the test.

In [ ]:
Type1_2=[]
for Office in Type1:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    Json=Clova2(Year,Page)
    
    try:
        XCoord_Unit=Detect_Office2(Json,Office)
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit+10
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row '+str(n))
        img=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
        #cv2.line(img, (XCoord_Unit,0), (XCoord_Unit,300), (255,0,0), 2)
        #cv2.imshow('pic',img)
        #cv2.waitKey(0)
    except:
        Type1_2.append(list((n,Dept,Office)))
        continue

#Step 6

Check the entire dataframe to confirm that the lines have been drawn at the correct place.

If there are errors, add the pair into the Type1 Error list and re-run step 5.

In [ ]:
Check(df)

In [ ]:
json_object = json.dumps(dta, indent=4,
                        cls=NpEncoder)
save_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+str(Year)+'\\'
with open(save_path+"DataFrame.json", "w") as outfile:
    outfile.write(json_object)